# smart-commit LoRA training on Kaggle

1. Settings: **Accelerator: GPU T4 x2**, **Internet: on** (needs a verified phone).
2. **Add Input** → the private dataset made by `sc-training kaggle-bundle train`.
3. **Save Version → Save & Run All (Commit)**: it runs in the background, up to 12 h.
4. Download `/kaggle/working/<run>.tar.gz` from the version's **Output**, then
   `tar xzf` it into `training/runs/` and continue with `export_gguf.py` locally.

Runs go one per GPU, in the order of `runs.json` (put the longest first);
each is packed as soon as it ends. Keep the whole bundle under ~11 h.

In [ ]:
import glob, json, os, pathlib, subprocess
bundle = pathlib.Path(glob.glob('/kaggle/input/**/runs.json', recursive=True)[0]).parent
spec = json.loads((bundle / 'runs.json').read_text())
print(bundle, json.dumps(spec, indent=1))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# Our locked versions; Kaggle's torch (and its CUDA) is kept.
!pip install -q -r {bundle}/requirements.txt
import torch, transformers, peft, bitsandbytes
print(torch.__version__, torch.version.cuda, transformers.__version__, peft.__version__, bitsandbytes.__version__)

In [ ]:
from huggingface_hub import snapshot_download
bases = {}
for run in spec['runs']:
    b = run['base']
    key = f"{b['repo']}@{b['revision']}"
    if key not in bases:
        bases[key] = snapshot_download(b['repo'], revision=b['revision'],
                                       local_dir=f"/kaggle/temp/{b['repo'].split('/')[1]}")
print(bases)

In [ ]:
import hashlib, time
work = pathlib.Path('/kaggle/working')
gpus = torch.cuda.device_count()
assert gpus, 'no GPU: turn the accelerator on in the notebook settings'
pending, running = list(spec['runs']), []
while pending or running:
    busy = {gpu for gpu, _, _ in running}
    for gpu in range(gpus):
        if pending and gpu not in busy:
            run = pending.pop(0)
            examples = bundle / run['examples']
            assert hashlib.sha256(examples.read_bytes()).hexdigest() == run['sha256'], run['name']
            out = work / 'runs' / run['name']
            out.mkdir(parents=True, exist_ok=True)
            log = open(work / f"{run['name']}.log", 'w')
            base = bases[f"{run['base']['repo']}@{run['base']['revision']}"]
            cmd = ['python', '-I', str(bundle / 'train.py'), str(examples), base, str(out), *run['args']]
            if run.get('val'):
                cmd += ['--val', str(bundle / run['val'])]
            print(f"gpu {gpu}: {' '.join(cmd)}", flush=True)
            proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                    env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)})
            running.append((gpu, run, proc))
            busy.add(gpu)
    time.sleep(30)
    for item in list(running):
        gpu, run, proc = item
        if proc.poll() is not None:
            running.remove(item)
            print(f"{run['name']}: exit {proc.returncode}", flush=True)
            print(open(work / f"{run['name']}.log").read()[-1500:], flush=True)
            # Packed right away, so a session that dies later keeps it.
            name = run['name']
            if (work / 'runs' / name / 'adapter').is_dir():
                packed = subprocess.run(['tar', 'czf', str(work / f'{name}.tar.gz'), '-C', str(work / 'runs'),
                                f'{name}/adapter', f'{name}/run.json', '-C', str(work), f'{name}.log'])
                if packed.returncode:
                    print(f'{name}: could not be packed, exit {packed.returncode}', flush=True)

In [ ]:
# Checkpoints are not kept: only the archives are worth downloading.
!rm -rf {work}/runs/*/checkpoints
!ls -la {work}